# Sheep 5′ RACE technical-primer trimming — cutadapt

`PRJNA900592` is a SMARTer 5′ RACE dataset, not multiplex V-primer PCR. There are no V-gene-specific forward primers to remove with MaskPrimers. However raw read starts are dominated by SMARTer/UPM-derived anchor and chain-specific C-region reverse primers from Park et al. Table 1. This separate stage removes those technical 5′ RACE primer sequences from the adapter-trimmed FASTQ and writes `results/<dataset>/pr_trimmed/fastq`. No QC is generated here.


In [ ]:
import os, sys, sysconfig, shutil, subprocess
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [_CONDA_ENV + "/lib/python3.11/site-packages", sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
print("cutadapt", shutil.which("cutadapt"))



In [ ]:
from pathlib import Path

# PRJNA900592 sheep is a 5′ RACE dataset, not multiplex V-primer PCR.
# There are no V-gene-specific forward primers to cut.
# But the raw 5′ ends are dominated by technical 5′ RACE primers:
# - SMARTer/UPM-derived 5′ anchor detected empirically by fastp and prefix counting.
# - Chain-specific reverse C-region primers from Park et al. Table 1.
# These are technical primer/linker sequences, so this separate stage removes them from read starts.
ADAPTER_MIN_OVERLAP = 10
FIVE_RACE_TECHNICAL_PRIMERS = [
    # SMARTer 5′ RACE / UPM-derived sequence, detected in PRJNA900592 raw reads.
    'CTAATACGACTCACTATAGGGCAAGCAGTGGTATCAACGCAGAGT',
    # Table 1, Park et al. 2023: gene-specific reverse primers near sheep constant regions.
    'TCCCCGCAGCAAGAAGTCAGAGGGTAG',      # Sh_IGHG_rev_1
    'GTTTGAAGAGGAAGACGGATGGCTGAGC',    # Sh_IGKC_rev_1
    'AGGGTGACCGAGGGTGCGGACTT',         # Sh_IGLC_rev_1
]



In [ ]:
def run_primer_trim_sheep(volume, dataset='PRJNA900592'):
    vol = Path(volume)
    src = vol / 'results' / dataset / 'trimmed' / 'fastq'
    base = vol / 'results' / dataset / 'pr_trimmed'
    out = base / 'fastq'
    logdir = base / 'cutadapt_reports'

    if base.exists():
        shutil.rmtree(base)
    out.mkdir(parents=True, exist_ok=True)
    logdir.mkdir(parents=True, exist_ok=True)

    pairs = sorted(set(f.name.replace('.trim.fastq.gz', '').rsplit('_', 1)[0] for f in src.glob('*.trim.fastq.gz')))
    print(f'[primer_trim_sheep] {dataset}: {len(pairs)} pairs')
    print('[primer_trim_sheep] 5′ RACE technical primer trim only; no V-primer MaskPrimers stage')

    for bn in pairs:
        r1 = src / f'{bn}_1.trim.fastq.gz'
        r2 = src / f'{bn}_2.trim.fastq.gz'
        if not r1.exists() or not r2.exists():
            continue
        o1 = out / f'{bn}_1.pr.fastq.gz'
        o2 = out / f'{bn}_2.pr.fastq.gz'
        log = logdir / f'{bn}.cutadapt.log'

        ca = ['cutadapt', '--compression-level', '1', '-O', str(ADAPTER_MIN_OVERLAP)]
        for s in FIVE_RACE_TECHNICAL_PRIMERS:
            ca += ['-g', '^' + s, '-G', '^' + s]
        ca += ['-o', str(o1), '-p', str(o2), str(r1), str(r2)]

        print(f'  [{bn}] cutadapt 5′ technical primers ...')
        rr = subprocess.run(ca, capture_output=True, text=True)
        log.write_text(rr.stdout + '\n' + rr.stderr)
        if rr.returncode != 0:
            raise RuntimeError(f'cutadapt failed {bn}: {rr.stderr[:1200]}')

    n = len(list(out.glob('*.pr.fastq.gz')))
    print(f'[primer_trim_sheep] DONE: {n} primer-trimmed files')




### Run

Run after `adapter_trim_sheep.ipynb` has produced `results/PRJNA900592/trimmed/fastq`.


In [ ]:
run_primer_trim_sheep('/data/user/epishkin', 'PRJNA900592')
